<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B01%5D%20-%20Intro_No_Supervisados/%5B01%5D%20-%20Notebooks/E2_KMeans_Completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · K-Means completo - Introducción a los modelos no supervisados

## Introducción

**K-Means** agrupa puntos alrededor de unos centros que se van ajustando. Su bucle es simple:

1. Elige K y coloca K centros (con `k-means++`, más separados y mejor que al azar).
2. Asigna cada punto a su centro más cercano.
3. Recalcula cada centro como la media de su grupo (su **centroide**).
4. Repite 2-3 hasta que los centros no se mueven.

En este ejercicio escalamos, entrenamos `KMeans(k=3, init='k-means++')`, miramos los
centroides, y usamos el **método del codo** y el **Silhouette Score** para elegir K y describir
cada grupo.

## Objetivos del ejercicio

- Entrenar K-Means con `k-means++` sobre datos escalados.
- Interpretar los **centroides**.
- Elegir K con el **método del codo** (inercia) y el **Silhouette Score**.
- Describir cada cluster.

## Descripción del dataset (clientes sin etiqueta)

Imagina que tienes una base de clientes y quieres ofrecerles un descuento, pero **no hay
etiquetas**: nadie te ha dicho qué cliente es de qué tipo. El objetivo del aprendizaje no
supervisado es justo ese: **descubrir la estructura** que hay dentro de los datos.

Generamos un dataset **sintético y reproducible** con `generar_clientes` (autocontenido en
Colab). Cada fila es un cliente con estas variables:

| Variable | Tipo | Descripción |
|---|---|---|
| `gasto_anual` | numérica | Gasto total al año (€), escala de miles |
| `num_visitas` | numérica | Nº de visitas al año, escala de decenas |
| `ticket_medio` | numérica | Gasto medio por compra (€) |
| `antiguedad_meses` | numérica | Meses como cliente |
| `edad` | numérica | Edad del cliente |
| `usa_app` | binaria | 1 si usa la app |
| `tiene_tarjeta_fidelidad` | binaria | 1 si tiene tarjeta de fidelidad |
| `compra_online` | binaria | 1 si compra online |
| `recibe_newsletter` | binaria | 1 si recibe la newsletter |
| `devuelve_productos` | binaria | 1 si suele devolver productos |

> Fíjate en las **escalas tan distintas** (gasto en miles, visitas en decenas). Esto va a ser
> clave: en clustering, la distancia depende de la escala, así que **habrá que escalar**.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

### 2. Datos y escalado (paso obligatorio)

In [ ]:
import numpy as np
import pandas as pd

def generar_clientes(n=600, semilla=42):
    # Dataset sintetico y reproducible de clientes SIN ETIQUETA para segmentacion.
    # Por dentro hay 4 perfiles latentes que el modelo deberia redescubrir, pero NO los
    # exponemos: en aprendizaje no supervisado no hay target, solo buscamos estructura.
    rng = np.random.default_rng(semilla)
    # perfil: (gasto_anual, num_visitas, ticket_medio, antiguedad_meses, edad,
    #          p_app, p_fidelidad, p_online, p_newsletter, p_devuelve)
    perfiles = [
        (9000, 42, 230, 60, 46, 0.85, 0.90, 0.70, 0.60, 0.10),  # grandes clientes
        (1100,  6, 120, 22, 37, 0.40, 0.20, 0.55, 0.30, 0.10),  # ocasionales
        (3200, 36,  75, 44, 52, 0.50, 0.65, 0.60, 0.80, 0.55),  # cazaofertas
        (2400, 15, 165,  9, 30, 0.92, 0.40, 0.95, 0.50, 0.20),  # nuevos digitales
    ]
    pesos = [0.22, 0.33, 0.25, 0.20]
    seg = rng.choice(len(perfiles), size=n, p=pesos)

    filas = []
    for s in seg:
        g, v, t, a, e, pa, pf, po, pn, pdv = perfiles[s]
        filas.append([
            round(max(50, rng.normal(g, g * 0.22)), 2),    # gasto_anual (€)
            int(max(1, round(rng.normal(v, v * 0.30)))),    # num_visitas
            round(max(5, rng.normal(t, t * 0.22)), 2),      # ticket_medio (€)
            int(max(1, round(rng.normal(a, 12)))),          # antiguedad_meses
            int(np.clip(rng.normal(e, 8), 18, 85)),         # edad
            int(rng.random() < pa),                          # usa_app
            int(rng.random() < pf),                          # tiene_tarjeta_fidelidad
            int(rng.random() < po),                          # compra_online
            int(rng.random() < pn),                          # recibe_newsletter
            int(rng.random() < pdv),                         # devuelve_productos
        ])
    cols = ["gasto_anual", "num_visitas", "ticket_medio", "antiguedad_meses", "edad",
            "usa_app", "tiene_tarjeta_fidelidad", "compra_online", "recibe_newsletter",
            "devuelve_productos"]
    return pd.DataFrame(filas, columns=cols)

In [ ]:
df = generar_clientes(n=600, semilla=42)
X = df.to_numpy(dtype=float)
X_esc = StandardScaler().fit_transform(X)     # escalar SIEMPRE antes de medir distancias
print("Datos escalados:", X_esc.shape)

### 3. K-Means con k-means++ (K=3 para empezar)

In [ ]:
km = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=0)
labels = km.fit_predict(X_esc)
print("Tamaño de cada cluster:", np.bincount(labels))
print("Inercia (suma de distancias a su centroide):", round(km.inertia_, 1))

### 4. Los centroides

Un **centroide** es el punto promedio del grupo (resume el grupo, aunque no sea un cliente
real). Como entrenamos sobre datos escalados, deshacemos el escalado para leerlos en sus
unidades originales.

In [ ]:
centroides = StandardScaler().fit(X).inverse_transform(km.cluster_centers_)
pd.DataFrame(centroides.round(1), columns=df.columns,
             index=[f"centroide_{i}" for i in range(3)])

### 5. ¿Cuántos grupos? Método del codo (inercia)

La **inercia** es la suma de distancias de los puntos a su centroide. Baja siempre al subir K;
buscamos el **codo**: donde añadir más clusters ya casi no la reduce.

In [ ]:
Ks = range(2, 7)
inercias = []
for k in Ks:
    inercias.append(KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit(X_esc).inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(Ks), inercias, marker="o")
plt.title("Método del codo")
plt.xlabel("K (nº de clusters)")
plt.ylabel("Inercia")
plt.tight_layout()
plt.show()

### 6. Silhouette Score

El **Silhouette** mide si cada punto está más cerca de su propio cluster (a) que del vecino
más cercano (b). Va de -1 a 1: cuanto más cerca de 1, mejor separados están los grupos.

In [ ]:
sils = []
for k in Ks:
    lab = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit_predict(X_esc)
    sils.append(silhouette_score(X_esc, lab))

for k, s in zip(Ks, sils):
    print(f"K={k}: silhouette = {s:.3f}")

mejor_k = list(Ks)[int(np.argmax(sils))]
print(f"\nMejor K según silhouette: {mejor_k}")

### 7. Modelo final y visualización en 2D (PCA)

In [ ]:
km_final = KMeans(n_clusters=mejor_k, init="k-means++", n_init=10, random_state=0)
labels = km_final.fit_predict(X_esc)

X_2d = PCA(n_components=2, random_state=0).fit_transform(X_esc)
plt.figure(figsize=(7, 6))
for c in range(mejor_k):
    m = labels == c
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=15, alpha=0.7, label=f"cluster {c}")
plt.title(f"Clusters de K-Means (K={mejor_k}) vistos en 2D con PCA")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.tight_layout()
plt.show()

### 8. Describir cada cluster

In [ ]:
perfil = df.assign(cluster=labels).groupby("cluster").mean().round(1)
perfil["n_clientes"] = np.bincount(labels)
perfil

### Reflexión

1. ¿Coinciden el "codo" y el mejor silhouette en el mismo K? ¿Qué harías si no coinciden?
2. ¿Qué aporta `k-means++` frente a inicializar los centros al azar?
3. Mirando los centroides, ¿cómo describirías con tus palabras cada grupo?
4. ¿Por qué hemos escalado antes de entrenar?